In [1]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


device: mps


In [2]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print("hidden_size:", model.config.hidden_size)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
def pool_first_token(hidden_state):
    pooled = hidden_state[:, 0, :]
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)
    return pooled

@torch.no_grad()
def encode_sentences(texts, batch_size=128, max_length=128):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        outputs = model(**enc)
        pooled = pool_first_token(outputs.last_hidden_state)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


In [5]:
emb1 = encode_sentences(sent1)
emb2 = encode_sentences(sent2)

scores = np.sum(emb1 * emb2, axis=1)
threshold = 0.85
y_pred = (scores >= threshold).astype(int)

print("emb1 shape:", emb1.shape)
print("emb2 shape:", emb2.shape)
print("score range:", float(scores.min()), float(scores.max()))
print("done")


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 768)
emb2 shape: (408, 768)
score range: 0.8139357566833496 0.9993458986282349
done


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


{'accuracy': 0.6862745098039216, 'f1': 0.8134110787172012}
                precision    recall  f1-score   support

not_paraphrase       1.00      0.01      0.02       129
    paraphrase       0.69      1.00      0.81       279

      accuracy                           0.69       408
     macro avg       0.84      0.50      0.41       408
  weighted avg       0.78      0.69      0.56       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("idx:", i)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine_score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine_score:", float(scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


idx: 0
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
cosine_score: 0.937690019607544
true: 1 pred: 1
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
cosine_score: 0.9651895761489868
true: 0 pred: 1
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
cosine_score: 0.9854921102523804
true: 0 pred: 1
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: 

In [8]:
summary = {
    "dataset": "glue/mrpc",
    "split": "validation",
    "model": model_name,
    "device": str(device),
    "pooling": "first_token",
    "threshold": float(threshold),
    "num_examples": len(ds),
    "accuracy": float(acc),
    "f1": float(f1),
}
summary


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'distilbert-base-uncased',
 'device': 'mps',
 'pooling': 'first_token',
 'threshold': 0.85,
 'num_examples': 408,
 'accuracy': 0.6862745098039216,
 'f1': 0.8134110787172012}